# RetentionAI — 12. Production API (Stage 12b)

Wires every prior stage's tested artifact into one FastAPI service:

- Stage 6 pipeline (fit once at startup, via `run_stage6_split`)
- Stage 8 XGBoost champion (`ADR-009`, confirmed real-data winner across
  PR-AUC, Precision@K, and Recall@K)
- Stage 9 isotonic calibration + Mondrian conformal thresholds (calib
  split carved fresh from the test set here, matching Stage 9's real
  approach — Stage 6 never built a 3-way split)
- Stage 11 Thompson Sampling, **Redis-backed** — not the in-process class
  from Stage 11's offline replay. A live API needs state shared across
  worker processes, which in-process attributes can't provide
- Stage 12a counterfactual search, run as a background task so a
  prediction never waits on an explanation

Uses FastAPI's `TestClient`, which triggers the real `lifespan` startup —
this actually trains XGBoost and calibrates it, nothing here is mocked.
Real Redis (via Docker Desktop), not a mock.

**Deliberately not included:** drift monitoring. Reasonable future work,
but no monitoring module has been built or tested in this project yet —
adding one now would repeat the exact mistake `ADR-000` exists to
prevent.

In [1]:
import time
from collections import Counter

from fastapi.testclient import TestClient
from src.api.main import app
import redis

# start clean for this demo
r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
for key in r.keys("bandit:*") + r.keys("counterfactual:*"):
    r.delete(key)

client = TestClient(app)
client.__enter__()  # explicitly trigger lifespan startup -- without this (or a `with` block),
                     # state.model stays None across cell boundaries in a notebook
print("Client ready (lifespan startup trains XGBoost + calibrates for real).")

d:\Important\00-Projects\RetentionAI\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Client ready (lifespan startup trains XGBoost + calibrates for real).


## Health check, then a real prediction

In [2]:
sample_customer = {
    "tenure": 3, "MonthlyCharges": 85.0, "TotalCharges": 255.0, "SeniorCitizen": 0,
    "Contract": "Month-to-month", "InternetService": "Fiber optic",
    "OnlineSecurity": "No", "OnlineBackup": "No", "DeviceProtection": "No",
    "TechSupport": "No", "StreamingTV": "Yes", "StreamingMovies": "Yes",
    "PaymentMethod": "Electronic check", "gender": "Female", "Partner": "No",
    "Dependents": "No", "PhoneService": "Yes", "MultipleLines": "No", "PaperlessBilling": "Yes",
}

print(client.get("/health").json())
response = client.post("/predict", json=sample_customer)
print(response.json())

{'status': 'ok', 'model_loaded': True}
{'request_id': 'd4ab49c6-32e0-48dd-9980-db844ac1f387', 'calibrated_churn_probability': 0.4084506928920746, 'conformal_prediction_set': [0, 1], 'recommended_arm': 'control', 'counterfactual_status': 'pending'}


## Poll for the background-computed counterfactual

Watch for a real, known ambiguity here (`ADR-014` Decision Point 4): if
this customer was already predicted "retained" by the raw model, the
result reads `raw_changes: {}, flippable: true` — indistinguishable from
"needed zero changes to flip." A documented gap, not silently hidden.

In [3]:
request_id = response.json()["request_id"]
for _ in range(20):
    cf = client.get(f"/counterfactual/{request_id}").json()
    if cf.get("status") == "ready":
        break
    time.sleep(0.1)
print(cf)

{'status': 'ready', 'raw_changes': {}, 'flippable': True}


## Feedback loop: does the bandit actually move?

Send 30 "retained" outcomes for `discount`, then compare arm selection
frequency before and after — the real, live-system half of Thompson
Sampling, distinct from Stage 11's offline replay evaluation.

In [4]:
def arm_distribution(n=100):
    picks = [client.post("/predict", json=sample_customer).json()["recommended_arm"] for _ in range(n)]
    return Counter(picks)

before = arm_distribution(100)
print("Before feedback:", before)

for _ in range(30):
    client.post("/feedback/discount", params={"retained": True})

after = arm_distribution(100)
print("After 30 'discount retained' feedback events:", after)

Before feedback: Counter({'control': 35, 'discount': 33, 'technician': 32})
After 30 'discount retained' feedback events: Counter({'discount': 96, 'technician': 4})


## Stage 12b summary

- Every endpoint verified genuinely working via `TestClient`, against a
  real, live Redis instance — no mocking.
- **Champion model corrected to XGBoost**, matching `ADR-009`'s real,
  confirmed result. An earlier draft of this stage mistakenly reverted to
  Logistic Regression, citing a real-data result that never happened —
  caught before shipping, not after.
- The arm distribution shift above is the actual point of Redis-backed
  state: outcome feedback through `/feedback` immediately changes what
  `/predict` recommends next, without restarting the service.
- Counterfactual generation genuinely runs as a background task —
  `/predict` returns before the search completes, and polling
  `/counterfactual/{id}` catches up once it's done.
- `transform_customer_for_inference` reuses Stage 6's own
  `prepare_features`/`transform_new` directly, verified byte-identical
  in column structure to the batch training path — real train/serve-skew
  prevention, not just a stated intention.

**Known simplifications, stated plainly (see `ADR-014` for full detail):**
- Models train fresh at API startup rather than loading from a persisted
  registry (e.g. MLflow) — fine for this environment, a real difference
  from a hardened deployment.
- The `raw_changes: {}` / "already retained" ambiguity (Decision Point 4)
  — reproduced live in this run, not just described.
- Test suite and the live API would share the same Redis DB (db=0) in a
  real dev environment — acceptable for a demo, a real bug risk in
  production.

Next: Stage 12c — Docker, docker-compose, GitHub Actions CI/CD, and the
Streamlit dashboard. Docker Desktop is already running on your machine,
so unlike this project's sandbox environment, Stage 12c's config can
actually be executed and verified for real there — worth doing, not a
constraint to write around this time.